In [0]:
%sql
-- LEAD_DIM — lead/student dimension for Coaching Clients
-- Identity (lead_name), status, ownership (CSM), commercial attributes (contract, tier, avatar)
-- Absorbs the former gold.lead_info names lookup — one lead dimension, one notebook
CREATE OR REPLACE TABLE crm_ingestion.gold.lead_dim AS (
WITH leads_processed_deduped AS (
    SELECT
    *
    FROM (
        SELECT
            id AS lead_id,
            name AS lead_name,
            display_name AS lead_display_name,
            status_label,
            date_updated,
            bronze_insert_date,
            ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS rn
        FROM
            crm_ingestion.silver.leads_processed
        WHERE
            status_label = 'Coaching Client'
        ) 
        WHERE
            rn = 1
)
,
leads_processed_custom_deduped AS (
    SELECT
        *
    FROM (
        SELECT
            leads_processed_id AS lead_id,
            CSM AS csm_id, -- — customer success manager
            Contracted_Value, -- — sales context
            Tier, -- — account info
            Avatar, -- profile image
            bronze_insert_date,
            ROW_NUMBER() OVER (PARTITION BY leads_processed_id ORDER BY bronze_insert_date DESC) AS rn
        FROM
            crm_ingestion.silver.leads_processed_custom
    )
    WHERE
        rn = 1
)
SELECT
    l.lead_id,
    l.lead_name,
    l.lead_display_name,
    l.status_label,
    l.date_updated,
    l.bronze_insert_date,
    NULLIF(c.csm_id, '\\N') AS csm_id,
    NULLIF(c.Contracted_Value, '\\N') AS Contracted_Value,
    NULLIF(c.Tier, '\\N') AS Tier,
    NULLIF(c.Avatar, '\\N') AS Avatar
FROM
    leads_processed_deduped l
LEFT JOIN
    leads_processed_custom_deduped c
ON
    l.lead_id = c.lead_id
)
